# **Advanced RAG Assignment - Part 2**

## Setup

In [12]:
import os
import logging
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from langchain_community.chat_models import ChatLiteLLM
from langchain_core.prompts import ChatPromptTemplate

# Load environment variables from .env file
load_dotenv()

# Setup logging
logging.basicConfig(
    format='[%(asctime)s] p%(process)s {%(filename)s:%(lineno)d} %(levelname)s - %(message)s',
    level=logging.INFO
)
logger = logging.getLogger(__name__)

In [13]:
if os.getenv("GROQ_API_KEY"):
    logger.info("GROQ_API_KEY is set")
else:
    logger.warning("No API key found! Please set GROQ_API_KEY in your .env file")

[2026-03-04 16:44:41,205] p1841 {3541512351.py:2} INFO - GROQ_API_KEY is set


In [14]:
MODEL_ID = "groq/llama-3.3-70b-versatile"

llm = ChatLiteLLM(model=MODEL_ID, temperature=0)
logger.info(f"Using model: {MODEL_ID}")

[2026-03-04 16:44:41,224] p1841 {493975163.py:4} INFO - Using model: groq/llama-3.3-70b-versatile


In [15]:
CSV_PATH = Path('../../data/structured/daily_sales.csv').resolve()
TEXT_DIR = Path('../../data/unstructured/').resolve()

logger.info(f"CSV: {CSV_PATH} (exists: {CSV_PATH.exists()})")
logger.info(f"Text dir: {TEXT_DIR} (exists: {TEXT_DIR.exists()})")

[2026-03-04 16:44:41,245] p1841 {1461164775.py:4} INFO - CSV: /home/ubuntu/dsan6725/assignments/spring-2026-a03-alisonmanna/data/structured/daily_sales.csv (exists: True)
[2026-03-04 16:44:41,246] p1841 {1461164775.py:5} INFO - Text dir: /home/ubuntu/dsan6725/assignments/spring-2026-a03-alisonmanna/data/unstructured (exists: True)


## Step 1: Query Router

Function for classifying the question to decide which data source(s) to query.

In [16]:
def classify_query(question):
    """
    Route the question to the right data source
      - csv_only: numerical/analytical questions about sales
      - text_only: product descriptions and reviews
      - both: questions that need both sources
    """
    q = question.lower()

    csv_kws = [
        'revenue', 'sales', 'volume', 'region', 'total', 'highest', 'lowest',
        'category', 'units', 'sold', 'december', 'october', 'november',
        'selling', 'sells', 'performing', 'popular'
    ]
    text_kws = [
        'feature', 'review', 'description', 'specification', 'customer',
        'say', 'cleaning', 'quality', 'rating', 'rated', 'recommend', 'highly'
    ]
    has_csv = any(kw in q for kw in csv_kws)
    has_text = any(kw in q for kw in text_kws)

    if has_csv and has_text:
        return 'both'
    elif has_csv:
        return 'csv_only'
    elif has_text:
        return 'text_only'
    else:
        return 'both' # default to both if unsure

# check
test_questions = [
    "What was the total revenue for Electronics category in December 2024?",
    "Which region had the highest sales volume?",
    "What are the key features of the Wireless Bluetooth Headphones?",
    "What do customers say about the Air Fryer's ease of cleaning?",
    "Which product has the best customer reviews and how well is it selling?",
    "I want a product for fitness that is highly rated and sells well in the West region. What do you recommend?",
]
for q in test_questions:
    logger.info(f"{classify_query(q)} <- {q[:65]}")

[2026-03-04 16:44:41,258] p1841 {1701205813.py:41} INFO - csv_only <- What was the total revenue for Electronics category in December 2
[2026-03-04 16:44:41,259] p1841 {1701205813.py:41} INFO - csv_only <- Which region had the highest sales volume?
[2026-03-04 16:44:41,260] p1841 {1701205813.py:41} INFO - text_only <- What are the key features of the Wireless Bluetooth Headphones?
[2026-03-04 16:44:41,261] p1841 {1701205813.py:41} INFO - text_only <- What do customers say about the Air Fryer's ease of cleaning?
[2026-03-04 16:44:41,262] p1841 {1701205813.py:41} INFO - both <- Which product has the best customer reviews and how well is it se
[2026-03-04 16:44:41,262] p1841 {1701205813.py:41} INFO - both <- I want a product for fitness that is highly rated and sells well 


## Step 2: CSV Retrieval

For numerical/sales questions: load CSV with pandas & return useful aggregations as context

In [17]:
def retrieve_from_csv(question):
    """
    Load sales CSV, apply keyword-based filters, return aggregated summaries as string for the LLM
    """
    df = pd.read_csv(CSV_PATH)
    q = question.lower()

    # Filter based on question keywords and track what was filtered
    filtered = df.copy()
    filter_desc = []

    if 'december' in q or 'dec ' in q:
        filtered = filtered[filtered['date'].str.startswith('2024-12')]
        filter_desc.append("December 2024")
    if 'october' in q or 'oct ' in q:
        filtered = filtered[filtered['date'].str.startswith('2024-10')]
        filter_desc.append("October 2024")
    if 'november' in q or 'nov ' in q:
        filtered = filtered[filtered['date'].str.startswith('2024-11')]
        filter_desc.append("November 2024")
    if 'electronics' in q:
        filtered = filtered[filtered['category'] == 'Electronics']
        filter_desc.append("Electronics category")
    if 'west' in q:
        filtered = filtered[filtered['region'] == 'West']
        filter_desc.append("West region")
    if 'fitness' in q or 'sport' in q:
        filtered = filtered[filtered['category'].str.lower().isin(['sports', 'fitness', 'sports & fitness'])]
        filter_desc.append("Sports/Fitness category")

    summaries = []

    # If filters were applied, show filtered totals w explicit label so the LLM knows what they represent
    if len(filtered) < len(df):
        label = ', '.join(filter_desc) if filter_desc else 'custom filter'
        summaries.append(
            f"*** Filtered Summary: {label} ({len(filtered)} rows) ***\n"
            f"Total Revenue: ${filtered['total_revenue'].sum():,.2f}\n"
            f"Total Units Sold: {filtered['units_sold'].sum():,}"
        )

    # Regional breakdown (all data)
    region_agg = df.groupby('region')[['units_sold', 'total_revenue']].sum().sort_values('total_revenue', ascending=False)
    summaries.append(f"*** Regional Sales (all data) ***\n{region_agg.to_string()}")

    # Category breakdown (all data)
    cat_agg = df.groupby('category')[['units_sold', 'total_revenue']].sum().sort_values('total_revenue', ascending=False)
    summaries.append(f"*** Category Sales (all data) ***\n{cat_agg.to_string()}")

    # Product breakdown on filtered data
    product_agg = filtered.groupby('product_name')[['units_sold', 'total_revenue']].sum().sort_values('total_revenue', ascending=False)
    data_label = f"Filtered: {label}" if len(filtered) < len(df) else "All"
    summaries.append(f"*** Product Sales ({data_label}) ***\n{product_agg.head(20).to_string()}")

    logger.info(f"CSV retrieval: {len(filtered)} rows after filtering ({', '.join(filter_desc) or 'no filter'})")
    return '\n\n'.join(summaries)

## Step 3: Text Retrieval

For product/review questions: search text files by keyword matching

In [18]:
MAX_TEXT_CHARS = 8000

def retrieve_from_text(question):
    """
    Search unstructured product pages for relevant content (simple keyword matching on file content).
    fix: files are ranked by # of keyword matches so the most relevant ones appear first
    """
    q = question.lower()

    stop_words = {
        'what', 'are', 'the', 'how', 'does', 'do', 'is', 'it', 'for', 'and',
        'that', 'this', 'with', 'from', 'have', 'has', 'about', 'well', 'which',
        'want', 'best', 'good', 'very', 'say', 'product', 'give', 'its', 'their'
    }

    # Extract keywords - strip punctuation and possessives (e.g. "fryer's" -> "fryer")
    keywords = []
    for w in q.split():
        w = w.strip('?.,!')
        if w.endswith("'s"):
            w = w[:-2]
        if len(w) > 2 and w not in stop_words:
            keywords.append(w)
    logger.info(f"Text search keywords: {keywords}")

    # Score each file by num keyword matches, then sort mostrelevant first
    scored = []
    for filepath in TEXT_DIR.glob('*.txt'):
        content = filepath.read_text().lower()
        score = sum(kw in content for kw in keywords)
        if score > 0:
            scored.append((score, filepath))
    scored.sort(reverse=True)

    results = []
    for score, filepath in scored:
        results.append(f"*** {filepath.name} ***\n{filepath.read_text()[:2500]}")

    logger.info(f"Text retrieval: {len(results)} files matched")

    combined = '\n\n'.join(results)
    if not combined:
        return "No relevant product pages found"
    return combined[:MAX_TEXT_CHARS]

## Step 4: Answer Generation

Now combine retreived context and pass it to the LLM

In [19]:
system_prompt = (
    "You are a helpful e-commerce data analyst assistant. "
    "Use the following context to answer the question. "
    "The context may include sales CSV data and/or product page text. "
    "Be specific and cite numbers or product names where relevant.\n\n"
    "Context:\n{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{question}"),
])


def multi_source_pipeline(question):
    """
    Full pipeline: classify -> retrieve from source(s) -> generate answer
    """
    query_type = classify_query(question)
    logger.info(f"Query type: {query_type}")

    context_parts = []

    if query_type in ('csv_only', 'both'):
        csv_context = retrieve_from_csv(question)
        context_parts.append(f"--- Sales Data (CSV) ---\n{csv_context}")

    if query_type in ('text_only', 'both'):
        text_context = retrieve_from_text(question)
        context_parts.append(f"--- Product Pages ---\n{text_context}")

    context = '\n\n'.join(context_parts)
    logger.info(f"Total context length: {len(context)} chars")

    chain = prompt | llm
    response = chain.invoke({"context": context, "question": question})

    return {
        "question": question,
        "query_type": query_type,
        "answer": response.content,
    }

logger.info("multi_source_pipeline crfeated")

[2026-03-04 16:44:41,339] p1841 {98338823.py:44} INFO - multi_source_pipeline crfeated


## Run Test Questions

In [20]:
test_questions = [
    "What was the total revenue for Electronics category in December 2024?",
    "Which region had the highest sales volume?",
    "What are the key features of the Wireless Bluetooth Headphones?",
    "What do customers say about the Air Fryer's ease of cleaning?",
    "Which product has the best customer reviews and how well is it selling?",
    "I want a product for fitness that is highly rated and sells well in the West region. What do you recommend?",
]

results = []
for i, q in enumerate(test_questions, 1):
    logger.info(f"\n{'='*60}")
    logger.info(f"Question {i}: {q}")
    result = multi_source_pipeline(q)
    results.append(result)
    logger.info(f"Answer preview: {result['answer'][:200]}...")

[2026-03-04 16:44:41,356] p1841 {270588187.py:12} INFO - 
[2026-03-04 16:44:41,359] p1841 {270588187.py:13} INFO - Question 1: What was the total revenue for Electronics category in December 2024?
[2026-03-04 16:44:41,359] p1841 {98338823.py:20} INFO - Query type: csv_only
[2026-03-04 16:44:41,395] p1841 {3211499370.py:55} INFO - CSV retrieval: 46 rows after filtering (December 2024, Electronics category)
[2026-03-04 16:44:41,399] p1841 {98338823.py:33} INFO - Total context length: 1475 chars
16:44:41 - LiteLLM:INFO: utils.py:3898 - 
LiteLLM completion() model= llama-3.3-70b-versatile; provider = groq
[2026-03-04 16:44:41,407] p1841 {utils.py:3898} INFO - 
LiteLLM completion() model= llama-3.3-70b-versatile; provider = groq
16:44:41 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler
[2026-03-04 16:44:41,656] p1841 {utils.py:1630} INFO - Wrapper: Completed Call, calling success_handler
[2026-03-04 16:44:41,658] p1841 {270588187.py:16} INFO - Answer preview:

In [21]:
# SHow full results
for i, r in enumerate(results, 1):
    print(f"\n{'-'*60}")
    print(f"Q{i} [{r['query_type']}]: {r['question']}")
    print(f"\nAnswer:\n{r['answer']}")


------------------------------------------------------------
Q1 [csv_only]: What was the total revenue for Electronics category in December 2024?

Answer:
The total revenue for the Electronics category in December 2024 was $142,864.31.

------------------------------------------------------------
Q2 [csv_only]: Which region had the highest sales volume?

Answer:
The Central region had the highest sales volume, with 6,779 units sold. This is according to the Regional Sales data, where the Central region has the highest number of units sold compared to the other regions.

------------------------------------------------------------
Q3 [text_only]: What are the key features of the Wireless Bluetooth Headphones?

Answer:
The key features of the Wireless Bluetooth Headphones (SKU: ELEC001) are:

1. Active Noise Cancellation (ANC) with transparency mode
2. Bluetooth 5.2 for stable connectivity up to 30ft range
3. 40-hour battery life, with quick charge (10 min = 3 hours playback)
4. Premium

## Save results

In [22]:
output_path = Path('../../part2_results.txt')

with open(output_path, 'w') as f:
    for i, r in enumerate(results, 1):
        f.write(f"Question {i}: {r['question']}\n")
        f.write(f"Query Type: {r['query_type']}\n")
        f.write(f"\nAnswer:\n{r['answer']}\n")
        f.write(f"\n{'='*60}\n\n")

logger.info(f"Saved results to {output_path}")

[2026-03-04 16:44:46,046] p1841 {3368522003.py:10} INFO - Saved results to ../../part2_results.txt
